# 01 — Hello World (HTML)

The simplest possible example with `HtmlBuilderHandler`.

**What you learn here:**
- Subclass `HtmlBuilderHandler` and bind a dialect to the handler (decision 9).
- Populate the `source` Bag inside `main(self, root)` (decision 5).
- The three-phase lifecycle: `create()` → `build()` → `render()`.
- Inspect the *recipe* (`source.to_xml()`) and the final output (`render()`).
- Switch between `xml=True` (default, XML well-formed) and `xml=False` (idiomatic HTML5).

## 1. Define a page

A page is a subclass of `HtmlBuilderHandler` that implements `main(self, root)`.
The `root` parameter is the `source` Bag where the page recipe is written.

In [ ]:
from genro_builders.contrib.html import HtmlBuilderHandler


class HelloPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.h1("Hello World")
        body.p("My first page with genro-builders.")

## 2. `create()` — populate the source

`create()` calls `main(self.source)` and fills the source with the recipe.
The source is the user representation, preserved faithfully (decision 7).

In [ ]:
page = HelloPage()
page.create()

print(page.source.to_xml())

## 3. `build()` — materialize the source into `built`

`build()` produces the `built` Bag. For the base case without `@component` or iterate it is a 1:1 mirror.
The source/built distinction becomes important when components need to be expanded (decision 7).

In [ ]:
page.build()
print(page.built.to_xml())

## 4. `render()` — produce the final HTML

The dialect defines *how* to serialize the built bag. `HtmlBuilder._default_render_mode` is `"html"`.

In [ ]:
print(page.render())

## 5. Inline preview (Jupyter)

In [ ]:
from IPython.display import HTML

HTML(page.render())

## 6. Void tags and self-close style

HTML5 void tags (`img`, `br`, `hr`, `input`, ...) are emitted self-close XHTML-style by default: `<img src="x"/>`.
This keeps the document XML well-formed (useful for pipelines that mix HTML and SVG/XML).
With `xml=False` you get the idiomatic HTML5 form: `<img src="x">`.

In [ ]:
class LogoPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.img(src="logo.png", alt="Logo")
        body.br()
        body.p("Below the logo.")


logo = LogoPage()
logo.create()
logo.build()

print("xml=True  (default):", logo.render())
print("xml=False (HTML5):  ", logo.render(xml=False))

## 7. Booleans as JS literals

Attributes with value `True`/`False` are serialized as the strings `"true"`/`"false"`,
so client-side JS can consume them directly.
The value `None` is currently filtered upstream (see `tests/test_html_render.py`).

In [ ]:
class FormPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.input(type="text", disabled=True)
        body.input(type="checkbox", checked=False)


form = FormPage()
form.create()
form.build()
print(form.render())

## 8. Keyword-collision attributes: `_class` → `class`

Python does not accept `class` as a keyword argument, so we write `_class`. The renderer maps the underscore-prefix to the correct HTML name (same for `_for`).

In [ ]:
class StyledPage(HtmlBuilderHandler):
    def main(self, root):
        body = root.body()
        body.div("Important", _class="alert primary")


styled = StyledPage()
styled.create()
styled.build()
print(styled.render())